In [ ]:
import torch
import torch.nn as nn

from loka.models import BreakthroughModel
from loka.models import BreakthroughCNN

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)
        # two layers, see https://arxiv.org/html/1512.03385v1#S3.E2
        

    def forward(self, x):
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return torch.relu(x + y)

class BreakthroughResNet(BreakthroughModel):
    preprocess = BreakthroughCNN.preprocess          # same flat-features -> board conversion

    def __init__(self, n_extra=0, channels=32, blocks=2, board=5,
                 head_ch=64, hidden=115):
        super().__init__()
        in_ch = 2 + n_extra
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, channels, 3, padding=1, bias=False),     # stem
            nn.BatchNorm2d(channels), nn.ReLU(),
            *[ResBlock(channels) for _ in range(blocks)],             # tower
            nn.Conv2d(channels, head_ch, 1, bias=False),              # value head
            nn.BatchNorm2d(head_ch), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(head_ch * board * board, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

model_CNN = BreakthroughCNN()
model = BreakthroughResNet(channels=32, blocks=2)

cnn_total = sum(p.numel() for p in model_CNN.parameters() if p.requires_grad)
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'CNN params:\t{cnn_total:,}')
print(f'ResNet params:\t{pytorch_total_params:,}')

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


x = torch.randn(1, 50)  
for name, m in [("CNN", model_CNN), ("ResNet", model)]:
    m.eval()
    with FlopCounterMode(display=False) as fc:
        m(x)
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name}:\t{params:,} params\t{fc.get_total_flops() / 2:,.0f} MACs")

In [ ]:
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import numpy as np
import torch

from loka.models import BreakthroughResNet18, BreakthroughCNN, BreakthroughModel
from loka.data import load_breakthrough_states
from loka.geometrical_methods import herding_order, k_center_order

from tqdm import tqdm

import os
from dotenv import load_dotenv

load_dotenv()

SEED = 42
torch.manual_seed(SEED)

BASE_DIR = os.getenv("BASE_DIR")

data_dir = f"{BASE_DIR}/data/game_states/e.txt"

device = "cuda" if torch.cuda.is_available() else "cpu"

X_np,Y_np = load_breakthrough_states(data_dir)
X,Y = torch.FloatTensor(X_np), torch.FloatTensor(Y_np)

X_3d = X.reshape(-1, 2, 5, 5)                           # (N, 2, 5, 5)
ones = torch.ones(X_3d.shape[0], 1, 5, 5, dtype=X.dtype, device=X.device)
X_3d = torch.cat([X_3d, ones], dim=1)

In [ ]:
cur_model = model_CNN.to(device)
optimizer = torch.optim.Adam(lr=1e-3, params=cur_model.parameters())
loss_fn = nn.MSELoss()

X_t,Y_t = X.to(device), Y.to(device)

with tqdm(total=50, desc="Training") as pbar:
    train(cur_model, optimizer, loss_fn, X_t[:100_000], Y_t[:100_000], 50, pbar, 1)

In [ ]:
@torch.inference_mode()
def evaluate_mse(model, X, Y, batch_size=4096):
    model.eval()
    device = next(model.parameters()).device
    total = torch.zeros((), device=device)
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        yb = Y[i:i + batch_size].to(device).reshape(-1).float()
        total += ((model(xb) - yb) ** 2).sum()
    model.train()
    return (total / len(X)).item()

n_train = 100_000
print(f"Train MSE (first {n_train:,}):  {evaluate_mse(cur_model, X[:n_train], Y[:n_train]):.6f}")
print(f"Held-out MSE (rest):          {evaluate_mse(cur_model, X[n_train:], Y[n_train:]):.6f}")
print(f"Full dataset MSE:             {evaluate_mse(cur_model, X, Y):.6f}")

In [ ]:
BATCH_SIZE = 4096
def train(model, optimizer, loss_fn, X, Y, epochs, pbar, pbar_inc):
    model.train()
    n = len(X)
    for _ in range(epochs):
        perm = torch.randperm(n, device=X.device)
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = X[idx], Y[idx]

            optimizer.zero_grad()
            batch_loss = loss_fn(model(xb), yb)
            batch_loss.backward()
            optimizer.step()

        pbar.update(pbar_inc)

In [ ]:
@torch.inference_mode()
def evaluate_mse(model, X, Y, batch_size=4096):
    model.eval()
    device = next(model.parameters()).device
    total = torch.zeros((), device=device)
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        yb = Y[i:i + batch_size].to(device).reshape(-1).float()
        total += ((model(xb) - yb) ** 2).sum()
    model.train()
    return (total / len(X)).item()

n_train = 100_000
print(f"Train MSE (first {n_train:,}):  {evaluate_mse(cur_model, X[:n_train], Y[:n_train]):.6f}")
print(f"Held-out MSE (rest):          {evaluate_mse(cur_model, X[n_train:], Y[n_train:]):.6f}")
print(f"Full dataset MSE:             {evaluate_mse(cur_model, X, Y):.6f}")

In [ ]:
cur_model = model.to(device)
optimizer = torch.optim.Adam(lr=1e-3, params=cur_model.parameters())
loss_fn = nn.MSELoss()

X_t,Y_t = X.to(device), Y.to(device)

with tqdm(total=50, desc="Training") as pbar:
    train(cur_model, optimizer, loss_fn, X_t[:100_000], Y_t[:100_000], 50, pbar, 1)

In [ ]:
@torch.inference_mode()
def evaluate_mse(model, X, Y, batch_size=4096):
    model.eval()
    device = next(model.parameters()).device
    total = torch.zeros((), device=device)
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        yb = Y[i:i + batch_size].to(device).reshape(-1).float()
        total += ((model(xb) - yb) ** 2).sum()
    model.train()
    return (total / len(X)).item()

n_train = 100_000
print(f"Train MSE (first {n_train:,}):  {evaluate_mse(cur_model, X[:n_train], Y[:n_train]):.6f}")
print(f"Held-out MSE (rest):          {evaluate_mse(cur_model, X[n_train:], Y[n_train:]):.6f}")
print(f"Full dataset MSE:             {evaluate_mse(cur_model, X, Y):.6f}")